# 01 — Prior Predictive Check

Sample parameters from the prior defined in `config.py` and run SIPNET in
parallel via PyEns. Verify that the prior is appropriately wide and that the
ground truth lies well within the prior predictive envelope.

**Pre-requisite:** run `python config.py` in the experiment directory to
generate `data/climate.clim` and `data/obs_nee.npy`.


In [ ]:
import notebook_env  # noqa: F401 — sets sys.path for config import
import numpy as np
import matplotlib.pyplot as plt

from config import build_prior, load_model, GROUND_TRUTH, DATA_DIR
from sipnet_calibration import prior_predictive
from sipnet_calibration.plotting import fan_chart


In [ ]:
# Load the model (reads climate + obs from data/)
_prob_model, obs_nee = load_model()

# Build the prior and extract the base SIPNETModel from the likelihood
prior = build_prior()
param_names = list(prior.record_template.fields)
print("Calibrated parameters:", param_names)
print("Prior flat size (MCMC dimensionality):", prior.record_template.flat_size)


In [ ]:
# We need just the SIPNETModel, not the full ProbPipe model
# Re-create it so we can pass it to prior_predictive
from pysipnet import SIPNETRunner, SIPNETModel
from pysipnet.climate import ClimateDrivers
from pysipnet.runner import ClimateStaging
from sipnet_calibration import default_base_params

climate = ClimateDrivers.from_path(str(DATA_DIR / "climate.clim"))
sipnet_model = SIPNETModel(
    SIPNETRunner(climate_staging=ClimateStaging.SYMLINK),
    base_params=default_base_params(),
    base_climate=climate,
)



In [ ]:
prior_nee = prior_predictive(prior, sipnet_model, n_samples=200, n_workers=4, seed=0)
print(f"Prior predictive shape: {prior_nee.shape}")


In [ ]:
# Ground truth NEE for visual reference
truth_nee = sipnet_model(**GROUND_TRUTH).nee().values
t = np.arange(prior_nee.shape[1])

ax = fan_chart(
    prior_nee,
    t=t,
    obs=obs_nee,
    truth=truth_nee,
    ylabel="NEE (gC m$^{-2}$ per 3-hr step)",
    title="Prior predictive fan chart",
)
plt.tight_layout()
plt.show()

print(f"Ground truth NEE range: [{truth_nee.min():.2f}, {truth_nee.max():.2f}]")
print(f"Prior 5–95% range: [{np.percentile(prior_nee, 5):.2f}, {np.percentile(prior_nee, 95):.2f}]")


In [ ]:
# Marginal prior distributions
import jax
key = jax.random.PRNGKey(99)
prior_samples = prior._sample(key, (2000,))

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.ravel()

for i, name in enumerate(param_names):
    ax = axes[i]
    vals = np.array(prior_samples[name])
    ax.hist(vals, bins=40, density=True, alpha=0.7, color="steelblue")
    ax.axvline(GROUND_TRUTH[name], color="red", lw=2, label="truth")
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("Value", fontsize=8)

axes[0].legend()
plt.suptitle("Prior marginal distributions  (red = ground truth)")
plt.tight_layout()
plt.show()
